# Onboarding — First-time setup for az-ml-experiments

Run this notebook **once** after cloning the repo onto an AML compute instance.
It validates your environment, confirms Key Vault and ADLS access, uploads the
country crosswalk to ADLS, and prints a checklist of any remaining manual steps.

**Prerequisites (set before running):**

In AML Studio → Compute → your instance → Edit → Environment variables, add:

| Variable | Value |
|---|---|
| `ADLS_ACCOUNT_NAME` | Your ADLS storage account name |
| `ADLS_CONTAINER` | `data` (or your container name) |
| `KEY_VAULT_URL` | `https://<vault-name>.vault.azure.net` |
| `AZURE_SUBSCRIPTION_ID` | Your subscription ID |
| `AZURE_RESOURCE_GROUP` | Your resource group name |
| `AZUREML_WORKSPACE_NAME` | Your AML workspace name |

Then restart the compute instance kernel before running this notebook.

In [ ]:
import os
import sys
from pathlib import Path

# Make utils/ importable from setup/ directory
REPO_ROOT = Path().resolve().parent
sys.path.insert(0, str(REPO_ROOT))

PASS = "\u2705"  # ✅
FAIL = "\u274c"  # ❌
WARN = "\u26a0\ufe0f"  # ⚠️

issues = []

## 1 — Validate environment variables

In [ ]:
REQUIRED_VARS = [
    "ADLS_ACCOUNT_NAME",
    "KEY_VAULT_URL",
    "AZURE_SUBSCRIPTION_ID",
    "AZURE_RESOURCE_GROUP",
    "AZUREML_WORKSPACE_NAME",
]

OPTIONAL_VARS = [
    ("ADLS_CONTAINER", "data"),
    ("AZUREML_TRAINING_RUN_ID", "(set after training)"),
]

print("Required environment variables:")
for var in REQUIRED_VARS:
    val = os.environ.get(var, "")
    if val:
        # Mask most of the value for display
        masked = val[:4] + "***" if len(val) > 4 else "***"
        print(f"  {PASS} {var} = {masked}")
    else:
        print(f"  {FAIL} {var} — NOT SET")
        issues.append(f"Missing env var: {var}")

print("\nOptional environment variables:")
for var, default in OPTIONAL_VARS:
    val = os.environ.get(var, "")
    status = PASS if val else WARN
    display = (val[:4] + "***") if val and len(val) > 4 else (val or f"(not set, default: {default})")
    print(f"  {status} {var} = {display}")

## 2 — Validate Key Vault access

In [ ]:
REQUIRED_SECRETS = [
    ("acled-api-key",             "ACLED API key — register at acleddata.com"),
    ("acled-email",               "ACLED registered email"),
    ("gcp-service-account-json",  "GCP service account JSON — from GCP IAM"),
    ("gcp-project-id",            "GCP project ID — from GCP Console"),
]

if not os.environ.get("KEY_VAULT_URL"):
    print(f"{FAIL} KEY_VAULT_URL not set — skipping Key Vault check")
else:
    try:
        from utils.keyvault import get_kv_client
        kv = get_kv_client()
        existing = {s.name for s in kv.list_properties_of_secrets()}
        print(f"{PASS} Key Vault connected: {os.environ['KEY_VAULT_URL']}")
        print(f"  {len(existing)} secrets found in vault.\n")
        print("Required secrets:")
        for name, description in REQUIRED_SECRETS:
            if name in existing:
                print(f"  {PASS} {name}")
            else:
                print(f"  {FAIL} {name} — MISSING  ({description})")
                issues.append(f"Missing KV secret: {name}")
    except Exception as e:
        print(f"{FAIL} Key Vault connection failed: {e}")
        issues.append(f"Key Vault access error: {e}")

## 3 — Validate ADLS write access

In [ ]:
ADLS_ACCOUNT = os.environ.get("ADLS_ACCOUNT_NAME", "")
ADLS_CONTAINER = os.environ.get("ADLS_CONTAINER", "data")

if not ADLS_ACCOUNT:
    print(f"{FAIL} ADLS_ACCOUNT_NAME not set — skipping ADLS check")
else:
    try:
        import adlfs
        from azure.identity import DefaultAzureCredential

        fs = adlfs.AzureBlobFileSystem(
            account_name=ADLS_ACCOUNT, credential=DefaultAzureCredential()
        )

        # Write a small test blob
        test_path = f"{ADLS_CONTAINER}/setup_test/ping.txt"
        with fs.open(test_path, "w") as f:
            f.write("ping")

        # Read it back
        with fs.open(test_path, "r") as f:
            content = f.read()
        assert content == "ping"

        # Delete it
        fs.rm(test_path)

        print(f"{PASS} ADLS write/read/delete OK")
        print(f"   account   : {ADLS_ACCOUNT}")
        print(f"   container : {ADLS_CONTAINER}")
    except Exception as e:
        print(f"{FAIL} ADLS access failed: {e}")
        print("   Check that the compute instance managed identity has")
        print("   'Storage Blob Data Contributor' on the storage account.")
        issues.append(f"ADLS access error: {e}")

## 4 — Upload country crosswalk to ADLS

The crosswalk CSV is committed to the repo and is read by notebooks directly via
relative path. It also needs to live in ADLS so the sweep job's `uri_file` input
can reference it (`jobs/sweep_outcome.yml`).

In [ ]:
CROSSWALK_LOCAL = REPO_ROOT / "data" / "country_crosswalk.csv"
CROSSWALK_ADLS  = "data/country_crosswalk.csv"

if not ADLS_ACCOUNT:
    print(f"{FAIL} ADLS_ACCOUNT_NAME not set — skipping crosswalk upload")
elif not CROSSWALK_LOCAL.exists():
    print(f"{FAIL} Local crosswalk not found at {CROSSWALK_LOCAL}")
    issues.append("country_crosswalk.csv missing from repo")
else:
    try:
        import adlfs
        from azure.identity import DefaultAzureCredential

        fs = adlfs.AzureBlobFileSystem(
            account_name=ADLS_ACCOUNT, credential=DefaultAzureCredential()
        )
        dest = f"{ADLS_CONTAINER}/{CROSSWALK_ADLS}"

        # Skip if already uploaded
        if fs.exists(dest):
            print(f"{PASS} Country crosswalk already on ADLS: {dest}")
        else:
            fs.put(str(CROSSWALK_LOCAL), dest)
            print(f"{PASS} Uploaded country crosswalk → {dest}")
    except Exception as e:
        print(f"{FAIL} Crosswalk upload failed: {e}")
        issues.append(f"Crosswalk upload error: {e}")

## 5 — Onboarding checklist

In [ ]:
print("=" * 60)
if not issues:
    print(f"{PASS} All automated checks passed.")
else:
    print(f"{FAIL} {len(issues)} issue(s) found:")
    for issue in issues:
        print(f"   • {issue}")

print()
print("Manual steps still required:")
print()

print("1. ACLED credentials — if not yet in Key Vault:")
print("   Register at https://acleddata.com/register, then:")
print("   az keyvault secret set --vault-name <vault> --name acled-api-key --value <key>")
print("   az keyvault secret set --vault-name <vault> --name acled-email   --value <email>")
print()

print("2. GCP credentials — if not yet in Key Vault:")
print("   GCP Console → IAM → Service Accounts → create key → Download JSON")
print("   az keyvault secret set --vault-name <vault> --name gcp-service-account-json \\")
print("     --file /path/to/service_account.json")
print("   az keyvault secret set --vault-name <vault> --name gcp-project-id --value <project>")
print()

print("3. CNTS (commercial dataset, CQ Press / Databanks International):")
print("   Download cnts.dta (or .xlsx/.csv) from your subscription, then upload:")
ADLS_ACCOUNT_DISPLAY = os.environ.get("ADLS_ACCOUNT_NAME", "<ADLS_ACCOUNT_NAME>")
print(f"   az storage blob upload \\")
print(f"     --account-name {ADLS_ACCOUNT_DISPLAY} --container-name data \\")
print(f"     --name source_data/cnts/cnts.dta --file /path/to/cnts.dta --auth-mode login")
print()

print("4. After running 03/02_train_xgboost_outcomes:")
print("   Paste the MLflow run ID into 04_inference/01_xgboost_inference.ipynb,")
print("   or set: export AZUREML_TRAINING_RUN_ID=<run-id>")
print()
print("=" * 60)
print("You are ready to run notebooks in stage order:")
print("  01_data_pull/ (any order) → 02_feature_engineering/ → 03_model_development/ → 04_inference/")